In [25]:
import xarray as xr

In [26]:
database_path = "contrib_gen_checkpoints/contribution_22_full.hdf5"

last = xr.open_dataset(database_path, engine="h5netcdf")


In [27]:
database_path = "contrib_gen_checkpoints/contribution_22_checkpoint_backup_1090.hdf5"

first = xr.open_dataset(database_path, engine="h5netcdf")

In [28]:
full = last.__deepcopy__()

In [29]:
full['contributions']

<xarray.DataArray 'contributions' (sample: 91392, species: 5, wavelength: 52)> Size: 190MB
[23761920 values with dtype=float64]
Coordinates:
  * wavelength  (wavelength) float64 416B 0.55 0.7 0.95 ... 6.381 6.814 7.276
  * sample      (sample) int64 731kB 0 1 2 3 4 ... 91387 91388 91389 91390 91391
  * species     (species) <U3 60B 'H2O' 'CO2' 'CH4' 'CO' 'NH3'
Attributes:
    units:        transit depth
    dataset:      taurex forward model
    description:  spectra per species if only that species was present in the...

In [30]:
# extract just the non-zero contribution coordinates from first

mask = first['contributions'].sum(dim='wavelength').sum(dim="species") > 0

first_nonzero = first.isel(sample=mask)

In [31]:
full['contributions'].loc[dict(sample=first_nonzero['sample'])] = first_nonzero['contributions']

In [32]:
full

<xarray.Dataset> Size: 329MB
Dimensions:                 (wavelength: 52, sample: 91392, parameter: 6,
                             quartile: 3, species: 5)
Coordinates:
  * wavelength              (wavelength) float64 416B 0.55 0.7 ... 6.814 7.276
  * sample                  (sample) int64 731kB 0 1 2 3 ... 91389 91390 91391
  * parameter               (parameter) <U13 312B 'planet_temp_k' ... 'log_NH3'
  * quartile                (quartile) <U2 24B 'q1' 'q2' 'q3'
  * species                 (species) <U3 60B 'H2O' 'CO2' 'CH4' 'CO' 'NH3'
Data variables: (12/22)
    bin_width               (wavelength) float64 416B ...
    spectrum                (sample, wavelength) float64 38MB ...
    noise                   (sample, wavelength) float64 38MB ...
    planet_temp_k           (sample) float64 731kB ...
    log_H2O                 (sample) float64 731kB ...
    log_CO2                 (sample) float64 731kB ...
    ...                      ...
    planet_radius_m         (sample) float64 731kB ...
    planet_surface_gravity  (sample) float64 731kB ...
    valid_quartiles         (sample) bool 91kB ...
    parameter_quartiles     (sample, parameter, quartile) float64 13MB ...
    contributions           (sample, species, wavelength) float64 190MB ...
    clean_forward_model     (sample, wavelength) float64 38MB ...

In [33]:
full.to_netcdf("contrib_gen_checkpoints/contribution_22_full.hdf5", engine="h5netcdf")